# 03 — Baseline Model

**ECE1513 Course Project — Traffic Congestion Prediction near U of T St. George Campus**

Before training complex ML models, we establish a simple baseline using a **Historical Average Model**. This model predicts the average speed for a given (location, hour, day_of_week) combination as the mean of all training observations with the same key. Any ML model we build must beat this baseline to be considered useful.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Processed Data

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
test_df = pd.read_csv('../data/processed/test.csv')

train_df['time_start'] = pd.to_datetime(train_df['time_start'])
test_df['time_start'] = pd.to_datetime(test_df['time_start'])

TARGET = 'avg_speed'
print(f'Target column: {TARGET}')
print(f'Train: {train_df.shape}, Test: {test_df.shape}')

## 2. Train Historical Average Model

The model computes the mean speed for each (location_id, hour_of_day, day_of_week) group in the training data. At prediction time, it looks up the group mean; if the group was not seen during training, it falls back to the global mean.

In [ ]:
# Group keys for the historical average
group_cols = ['location_id', 'hour_of_day', 'day_of_week']

# Compute group means from training data
group_means = train_df.groupby(group_cols)[TARGET].mean()
global_mean = train_df[TARGET].mean()

print(f'Number of unique groups: {len(group_means):,}')
print(f'Global mean speed: {global_mean:.2f} km/h')

# Predict on test set
test_keys = test_df[group_cols].apply(tuple, axis=1)
y_pred_baseline = test_keys.map(group_means).fillna(global_mean).values
y_true = test_df[TARGET].values

# How many test records had a matching group?
matched = test_keys.isin(group_means.index).sum()
print(f'Test records with matching group: {matched:,} / {len(test_df):,} ({matched/len(test_df)*100:.1f}%)')

## 3. Evaluate Baseline Performance

In [ ]:
mae = mean_absolute_error(y_true, y_pred_baseline)
rmse = np.sqrt(mean_squared_error(y_true, y_pred_baseline))
r2 = r2_score(y_true, y_pred_baseline)

print(f'Historical Average Baseline')
print(f'  MAE  = {mae:.3f} km/h')
print(f'  RMSE = {rmse:.3f} km/h')
print(f'  R2   = {r2:.4f}')

## 4. Predictions vs. Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_true, y_pred_baseline, alpha=0.1, s=8)
lims = [min(y_true.min(), y_pred_baseline.min()) - 5,
        max(y_true.max(), y_pred_baseline.max()) + 5]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Speed (km/h)')
axes[0].set_ylabel('Predicted Speed (km/h)')
axes[0].set_title('Baseline: Predicted vs Actual')
axes[0].legend()

# Residual distribution
residuals = y_true - y_pred_baseline
axes[1].hist(residuals, bins=60, edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (km/h)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Baseline: Residual Distribution')

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/baseline_predictions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Time-series snippet: actual vs predicted for a sample location
sample_loc = test_df['location_id'].mode().iloc[0]
snippet = test_df[test_df['location_id'] == sample_loc].head(500).copy()
snippet_keys = snippet[group_cols].apply(tuple, axis=1)
snippet['predicted'] = snippet_keys.map(group_means).fillna(global_mean).values

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(snippet['time_start'], snippet[TARGET], label='Actual', linewidth=0.8)
ax.plot(snippet['time_start'], snippet['predicted'], label='Baseline Predicted', linewidth=0.8, alpha=0.8)
ax.set_xlabel('Date / Time')
ax.set_ylabel('Speed (km/h)')
loc_name = snippet['location_name'].iloc[0] if 'location_name' in snippet.columns else f'Location {sample_loc}'
ax.set_title(f'Baseline: Actual vs Predicted — {loc_name}')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Baseline Error by Hour of Day

In [ ]:
test_df['residual_baseline'] = y_true - y_pred_baseline

error_by_hour = test_df.groupby('hour_of_day')['residual_baseline'].agg(
    MAE=lambda x: np.mean(np.abs(x)),
    Mean_Residual='mean'
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(error_by_hour['hour_of_day'], error_by_hour['MAE'], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('MAE (km/h)')
axes[0].set_title('Baseline MAE by Hour of Day')
axes[0].set_xticks(range(24))

axes[1].bar(error_by_hour['hour_of_day'], error_by_hour['Mean_Residual'], color='coral', edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Mean Residual (km/h)')
axes[1].set_title('Baseline Mean Residual (Bias) by Hour')
axes[1].set_xticks(range(24))

plt.tight_layout()
plt.show()

## 6. Discussion

The Historical Average baseline provides a useful reference point:

- **MAE** tells us the average prediction error in km/h. Even a simple average grouped by location, hour, and day captures the dominant daily traffic pattern on city streets.
- **R-squared** indicates how much variance in speed the baseline explains. Values significantly above zero confirm that temporal patterns are a strong signal for urban street speeds.
- **Limitations**: The baseline cannot react to weather conditions, special events, or short-term fluctuations. It also ignores interactions between variables (e.g., rain during rush hour being worse than rain at midnight).

In the next notebook we will train ML models (Linear Regression, Random Forest, XGBoost) and compare them against this baseline. A good model should reduce MAE meaningfully and capture the variance the baseline misses.